In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_csv('traffic_violations.csv')
print(df.shape, df.columns.tolist())
df.head()

In [ ]:
df = df.drop(columns=['Description','Charge','DL.State','Model'], errors='ignore')
df['Year'] = df['Year'].fillna(df['Year'].median())
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

# Target: 1 = Citation (violation), 0 = other
df_orig = pd.read_csv('traffic_violations.csv')
df['target'] = (df_orig['Violation.Type'].fillna('') == 'Citation').astype(int)

X = df.drop(columns=['target','Violation.Type'], errors='ignore')
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [1]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {'Accuracy': accuracy_score(y_test, y_pred), 'F1': f1_score(y_test, y_pred)}
    print(f"\n{name}:\n{classification_report(y_test, y_pred)}")

NameError: name 'RandomForestClassifier' is not defined

In [ ]:
names = list(results.keys())
accs = [results[n]['Accuracy']*100 for n in names]
plt.figure(figsize=(8,4))
bars = plt.bar(names, accs, color=['crimson','navy','steelblue','gray'])
plt.ylim(60, 95)
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy (%)')
for b, a in zip(bars, accs):
    plt.text(b.get_x()+b.get_width()/2, a+0.3, f'{a:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

In [ ]:
rf = models['Random Forest']
fi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
fi.plot(kind='barh', figsize=(8,5), color='steelblue', title='Top 10 Feature Importances')
plt.tight_layout(); plt.savefig('feature_importance.png', dpi=150); plt.show()

In [ ]:
df_orig2 = pd.read_csv('traffic_violations.csv')
df_orig2['is_violation'] = (df_orig2['Violation.Type'] == 'Citation')
risk = df_orig2.groupby('Driver.City')['is_violation'].sum().sort_values(ascending=False).head(10)
risk.plot(kind='barh', figsize=(8,5), color='crimson', title='Top 10 High-Risk Cities')
plt.tight_layout(); plt.savefig('risk_zones.png', dpi=150); plt.show()